# 08 — Jitter Quality Evaluation: EDD, MNND, and Cluster Fidelity

Three evaluation metrics from Lin (2023) applied to `render_coordinates()`:

1. **EDD** (Expected Displacement Distance) — mean haversine distance from each shuffled tile centre to its jittered display pin. Bounded by J√2 where J = `bin_size_m × jitter_max_frac`.
2. **MNND** (Mean Nearest-Neighbour Distance) — measures within-tile positional diversity of co-located display pins. Too small → co-location fingerprinting risk.
3. **DBSCAN F1** — pairwise cluster-membership agreement between true and display coordinate sets. In this library a *low* F1 is a privacy indicator: the PRP intentionally destroys geographic cluster structure.

**Key distinction from Lin (2023):** Lin measures *utility preservation* (high F1 = good masking). Here the PRP shuffle breaks spatial structure by design, so F1 is deliberately low. EDD and MNND are computed within the shuffled-tile space to isolate the jitter mechanism.

<div style="background:#f5faf9;border:1px solid #b8ddd8;border-radius:8px;padding:12px 14px 14px;margin:10px 0 22px;font-family:sans-serif;">
<div style="font-size:11px;color:#5a9e99;margin-bottom:10px;font-style:italic;">Quantitative privacy evaluation of the complete pipeline</div>
<div style="display:flex;align-items:stretch;">
    <div style="background:#2a9d8f;color:white;clip-path:polygon(0 0, calc(100% - 22px) 0, 100% 50%, calc(100% - 22px) 100%, 0 100%);padding:10px 16px 10px 16px;margin-left:0;position:relative;z-index:1;min-width:115px;text-align:center;"><div style="font-size:10px;opacity:0.85;margin-bottom:3px;">NB02</div><div style="font-weight:700;font-size:13px;">① Project</div></div>
    <div style="background:#2a9d8f;color:white;clip-path:polygon(22px 0, calc(100% - 22px) 0, 100% 50%, calc(100% - 22px) 100%, 22px 100%, 0 50%);padding:10px 16px 10px 36px;margin-left:-21px;position:relative;z-index:2;min-width:115px;text-align:center;"><div style="font-size:10px;opacity:0.85;margin-bottom:3px;">NB03</div><div style="font-weight:700;font-size:13px;">② Snap+Shuffle</div></div>
    <div style="background:#2a9d8f;color:white;clip-path:polygon(22px 0, calc(100% - 22px) 0, 100% 50%, calc(100% - 22px) 100%, 22px 100%, 0 50%);padding:10px 16px 10px 36px;margin-left:-21px;position:relative;z-index:3;min-width:115px;text-align:center;"><div style="font-size:10px;opacity:0.85;margin-bottom:3px;">NB04</div><div style="font-weight:700;font-size:13px;">③ Lock</div></div>
    <div style="background:#2a9d8f;color:white;clip-path:polygon(22px 0, 100% 0, 100% 100%, 22px 100%, 0 50%);padding:10px 16px 10px 36px;margin-left:-21px;position:relative;z-index:4;min-width:115px;text-align:center;"><div style="font-size:10px;opacity:0.85;margin-bottom:3px;">NB05</div><div style="font-weight:700;font-size:13px;">④ Wobble</div></div>
</div>
</div>

## Learning Objectives

By the end of this notebook you will be able to:

1. **Define** Expected Displacement Distance, Mean Nearest-Neighbour Distance, and DBSCAN cluster fidelity as formulated in Lin (2023).
2. **Interpret** the jitter-sweep results and explain why EDD and MNND scale linearly with `jitter_max_frac`.
3. **Calculate** EDD and MNND for the cholera dataset and compare values at three jitter levels.
4. **Compare** cluster fidelity for original versus display positions and explain why the PRP layer destroys cluster structure.
5. **Justify** which of the three metrics best captures epidemiological utility for a disease-outbreak investigation.

In [1]:
import secrets
import math
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import DBSCAN

from map_encryption import (
    MapEncryption, SchemeParams,
    _project, _unproject,
    haversine_m, edd, mnnd, dbscan_f1,
)

MASTER_KEY = secrets.token_bytes(32)  # in production: load from secrets manager
rng = np.random.default_rng(42)
CENTER_LAT, CENTER_LON = 51.513341, -0.136668  # Broadwick Street pump, Soho, London (1854 cholera outbreak)

## Part 1 — Jitter Quality on Co-Located Records

The display jitter mechanism defends against **co-location fingerprinting**: when many records share the same encrypted tile `(qxp, qyp)`, identical display pins would let an attacker spot them as co-located.

We evaluate by encoding 300 records at the same true location with the same tweak (→ same shuffled tile), then measuring how jitter spreads their display pins around the **shuffled tile centre** — the only reference point the display tier has.

In [2]:
enc = MapEncryption(MASTER_KEY, SchemeParams())
N_COLOC = 300
COLOC_TWEAK = MapEncryption.make_tweak(record_id=1, extra=b'nb08-coloc')

records = [
    enc.encode(CENTER_LAT, CENTER_LON, tweak=COLOC_TWEAK)
    for _ in range(N_COLOC)
]
assert len(set(r['qxp'] for r in records)) == 1
assert len(set(r['qyp'] for r in records)) == 1

B = enc.params.bin_size_m
J = B * enc.params.jitter_max_frac
tile_centers   = [_unproject(r['qxp'] * B, r['qyp'] * B) for r in records]
display_coords = [enc.render_coordinates(r) for r in records]

tile_lat, tile_lon = tile_centers[0]
print(f'Shuffled tile : qxp={records[0]["qxp"]}, qyp={records[0]["qyp"]}')
print(f'Tile centre   : {tile_lat:.4f}N  {tile_lon:.4f}E')
print(f'J = {J:.1f} m per axis   max diagonal = {J * 2**0.5:.1f} m')

Shuffled tile : qxp=-50645, qyp=-48452
Tile centre   : -72.9725N  -113.7379E
J = 62.5 m per axis   max diagonal = 88.4 m


In [3]:
import folium

m = folium.Map(location=[tile_lat, tile_lon], zoom_start=13, tiles='cartodbpositron')
for lat, lon in display_coords:
    folium.CircleMarker(
        location=[lat, lon],
        radius=4,
        color='steelblue',
        fill=True,
        fill_color='steelblue',
        fill_opacity=0.6,
    ).add_to(m)
folium.CircleMarker(
    location=[tile_lat, tile_lon],
    radius=12,
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.9,
    tooltip='shuffled tile centre',
    weight=3,
).add_to(m)
m

**Figure 8a.** Folium map of display-tier positions for all 250 cholera death records after full-pipeline encoding (PRP shuffle + jitter), showing how the encrypted display coordinates are scattered far from the original Soho cluster.

### EDD — Expected Displacement Distance

`edd(tile_centers, display_coords)` measures the mean distance from each shuffled tile centre to its jittered display pin.  
Jitter draws uniformly from [−J, +J] per axis, so the expected displacement is J · √(2/3) ≈ 0.82 J, and the hard maximum is J√2.

In [4]:
edd_val = edd(tile_centers, display_coords)
displacements = [
    haversine_m(t[0], t[1], d[0], d[1])
    for t, d in zip(tile_centers, display_coords)
]
max_diag = J * 2 ** 0.5

fig = px.histogram(
    x=displacements, nbins=30,
    labels={'x': 'Displacement from tile centre (metres)', 'y': 'Count'},
    title=f'EDD = {edd_val:.1f} m  |  theoretical max = {max_diag:.1f} m',
    template='plotly_white', height=400,
)
fig.add_vline(x=edd_val,  line_dash='dash', line_color='steelblue',
              annotation_text=f'EDD {edd_val:.1f} m')
fig.add_vline(x=max_diag, line_dash='dot',  line_color='red',
              annotation_text=f'max {max_diag:.1f} m')
fig.show()

assert max(displacements) <= max_diag + 1
print(f'EDD = {edd_val:.2f} m   min = {min(displacements):.2f} m   max = {max(displacements):.2f} m')
print('All displacements within theoretical bound: OK')

EDD = 14.25 m   min = 2.02 m   max = 25.15 m
All displacements within theoretical bound: OK


**Figure 8b.** Bar chart of Expected Displacement Distance (EDD, metres) for the jitter-only and full-pipeline scenarios, quantifying the average positional shift introduced by each privacy mechanism relative to true tile centres.

### MNND — Mean Nearest-Neighbour Distance

For 300 records at the **same** tile centre, `mnnd(tile_centers) ≈ 0`.  
The display MNND is therefore pure jitter diversity: the average gap between neighbouring display pins.  
A larger MNND means pins are more spread out → harder to fingerprint co-located records.

In [5]:
mnnd_tile = mnnd(tile_centers)
mnnd_disp = mnnd(display_coords)

fig = go.Figure(go.Bar(
    x=['Tile centres', 'Display positions'],
    y=[mnnd_tile, mnnd_disp],
    marker_color=['steelblue', 'coral'],
    text=[f'{mnnd_tile:.2f} m', f'{mnnd_disp:.2f} m'],
    textposition='outside',
))
fig.update_layout(
    title='MNND: tile centres vs display positions (co-location scenario)',
    yaxis_title='Mean nearest-neighbour distance (m)',
    template='plotly_white', height=400,
)
fig.show()
print(f'MNND (tile centres): {mnnd_tile:.2f} m')
print(f'MNND (display):      {mnnd_disp:.2f} m  ← jitter diversity')

MNND (tile centres): 0.00 m
MNND (display):      3.47 m  ← jitter diversity


**Figure 8c.** Bar chart of Mean Nearest-Neighbour Distance (MNND, metres) for the original, jitter-only, and full-pipeline coordinate sets, measuring how nearest-neighbour proximity is altered by each privacy mechanism.

## Part 2 — Privacy Demonstration: PRP Breaks Cluster Structure

With per-record tweaks (normal production use) each true tile maps to a different shuffled tile. Records that are geographically close in the true space land at independent, pseudorandom positions in the display space.

DBSCAN on display coordinates should recover **no correspondence** with true clusters. A low F1 is the expected and desired result.

In [6]:
import pandas as pd
from collections import Counter

# Load actual 1854 Soho cholera data
deaths = pd.read_csv('data/cholera_deaths.csv')
pumps_df = pd.read_csv('data/pumps.csv')

true_coords = list(zip(deaths['LAT'].values, deaths['LON'].values))

# Assign each death location to its nearest water pump (natural geographic cluster).
# The Broadwick Street pump dominates — this is the cluster structure Dr. John Snow
# identified by mapping deaths onto the street grid.
pump_locs  = list(zip(pumps_df['LAT'].values, pumps_df['LON'].values))
pump_names = pumps_df['Street'].tolist()

true_labels = []
for lat, lon in true_coords:
    dists = [haversine_m(lat, lon, plat, plon) for plat, plon in pump_locs]
    true_labels.append(pump_names[int(np.argmin(dists))])

mc_records = [
    enc.encode(lat, lon, tweak=MapEncryption.make_tweak(i))
    for i, (lat, lon) in enumerate(true_coords)
]
mc_display = [enc.render_coordinates(r) for r in mc_records]

counts = Counter(true_labels)
print(f'{len(true_coords)} cholera death locations assigned to {len(pump_locs)} pump catchment areas:')
for pump, n in counts.most_common():
    print(f'  {pump:<22} {n:>3} deaths')


250 cholera death locations assigned to 8 pump catchment areas:
  Broadwick Street       171 deaths
  Kingly Street           36 deaths
  Rupert Street           19 deaths
  Bridle Lane             13 deaths
  Warwick Street           9 deaths
  Ramillies Place          2 deaths


In [7]:
from IPython.display import display as ipy_display
import folium

# Colour-code deaths by their nearest pump.
# Broadwick Street pump (red) is the dominant cluster — John Snow's key finding.
pump_colors = {
    'Broadwick Street': 'red',
    'Kingly Street':    'blue',
    'Ramillies Place':  'green',
    'Dean Street':      'orange',
    'Rupert Street':    'purple',
    'Bridle Lane':      'darkblue',
    'Regent Street':    'cadetblue',
    'Warwick Street':   'darkred',
}

# Map 1: true geographic clusters — deaths coloured by nearest pump (Soho area)
m_true = folium.Map(location=[51.512, -0.136], zoom_start=15, tiles='cartodbpositron')
for (lat, lon), label in zip(true_coords, true_labels):
    folium.CircleMarker(
        location=[lat, lon],
        radius=5,
        color=pump_colors.get(label, 'gray'),
        fill=True,
        fill_color=pump_colors.get(label, 'gray'),
        fill_opacity=0.7,
        tooltip=f'Nearest pump: {label}',
    ).add_to(m_true)
ipy_display(m_true)

# Map 2: display positions after PRP shuffle — cluster structure destroyed
clat_d = sum(c[0] for c in mc_display) / len(mc_display)
clon_d = sum(c[1] for c in mc_display) / len(mc_display)
m_disp = folium.Map(location=[clat_d, clon_d], zoom_start=2, tiles='cartodbpositron')
for (lat, lon), label in zip(mc_display, true_labels):
    folium.CircleMarker(
        location=[lat, lon],
        radius=5,
        color=pump_colors.get(label, 'gray'),
        fill=True,
        fill_color=pump_colors.get(label, 'gray'),
        fill_opacity=0.7,
        tooltip=f'{label} (display)',
    ).add_to(m_disp)
m_disp


**Figure 8e.** Left: Folium map of 250 cholera death locations colour-coded by nearest water pump catchment (Broadwick Street in red, other pumps in distinct colours), showing the natural geographic cluster structure at Soho zoom. Right: display-tier positions after Feistel PRP shuffle (world zoom), demonstrating that the cluster colour pattern is completely destroyed — confirming that the PRP breaks spatial structure as intended.

In [8]:
EPS_M, MIN_SAMPLES = 400, 5

true_xy = np.array([_project(lat, lon) for lat, lon in true_coords])
disp_xy = np.array([_project(lat, lon) for lat, lon in mc_display])

true_db = DBSCAN(eps=EPS_M, min_samples=MIN_SAMPLES).fit_predict(true_xy)
disp_db = DBSCAN(eps=EPS_M, min_samples=MIN_SAMPLES).fit_predict(disp_xy)

print(f'DBSCAN on true coords:    {len(set(true_db) - {-1})} clusters, '
      f'{(true_db == -1).sum()} noise')
print(f'DBSCAN on display coords: {len(set(disp_db) - {-1})} clusters, '
      f'{(disp_db == -1).sum()} noise')

metrics = dbscan_f1(true_coords, mc_display, eps_m=EPS_M, min_samples=MIN_SAMPLES)
print('\nDBSCAN F1 — true vs display:')
for k, v in metrics.items():
    print(f'  {k}: {v}')
print('\nLow F1 confirms the PRP has destroyed geographic cluster structure (intended behaviour).')

DBSCAN on true coords:    1 clusters, 0 noise
DBSCAN on display coords: 0 clusters, 250 noise

DBSCAN F1 — true vs display:
  precision: 0.0
  recall: 0.0
  f1: 0.0
  n_true_clusters: 1
  n_display_clusters: 0
  n_noise_true: 0
  n_noise_display: 250

Low F1 confirms the PRP has destroyed geographic cluster structure (intended behaviour).


## Part 3 — Jitter Sweep: EDD and MNND vs `jitter_max_frac`

Using the co-location scenario (300 records, same shuffled tile), sweep `jitter_max_frac` from 0.05 to 2.0:

- **EDD** increases linearly — directly controlled by J.
- **Display MNND** shows how within-tile positional diversity scales with jitter.  
  Small MNND → pins still cluster → fingerprinting risk.  
  Large MNND → good diversity, but display pins may bleed into adjacent tiles visually.

The default `jitter_max_frac = 0.25` (J = 62.5 m) is marked on both plots.

In [9]:
fracs = [0.05, 0.10, 0.25, 0.50, 1.00, 2.00]
sweep = []

for frac in fracs:
    enc_i  = MapEncryption(MASTER_KEY, SchemeParams(jitter_max_frac=frac))
    disp_i = [enc_i.render_coordinates(r) for r in records]
    tc_i   = [_unproject(r['qxp'] * B, r['qyp'] * B) for r in records]

    edd_i    = edd(tc_i, disp_i)
    mnnd_d_i = mnnd(disp_i)
    J_i      = B * frac

    sweep.append({'frac': frac, 'J_m': J_i, 'edd_m': edd_i, 'mnnd_disp_m': mnnd_d_i})
    print(f'frac={frac:.2f}  J={J_i:5.1f} m  EDD={edd_i:6.1f} m  MNND_disp={mnnd_d_i:6.2f} m')

df_sweep = pd.DataFrame(sweep)

frac=0.05  J= 12.5 m  EDD=   2.8 m  MNND_disp=  0.69 m
frac=0.10  J= 25.0 m  EDD=   5.7 m  MNND_disp=  1.39 m
frac=0.25  J= 62.5 m  EDD=  14.2 m  MNND_disp=  3.47 m
frac=0.50  J=125.0 m  EDD=  28.5 m  MNND_disp=  6.95 m


frac=1.00  J=250.0 m  EDD=  57.0 m  MNND_disp= 13.90 m
frac=2.00  J=500.0 m  EDD= 114.0 m  MNND_disp= 27.79 m


In [10]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['EDD vs jitter_max_frac', 'Display MNND vs jitter_max_frac'],
)

fig.add_trace(go.Scatter(
    x=df_sweep['frac'], y=df_sweep['edd_m'],
    mode='lines+markers', name='EDD (m)',
    line=dict(color='coral', width=2), marker=dict(size=8),
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_sweep['frac'], y=df_sweep['mnnd_disp_m'],
    mode='lines+markers', name='MNND display (m)',
    line=dict(color='steelblue', width=2), marker=dict(size=8),
), row=1, col=2)

for col in [1, 2]:
    fig.add_vline(x=0.25, line_dash='dash', line_color='gray',
                  annotation_text='default 0.25',
                  annotation_position='top right',
                  row=1, col=col)

fig.update_xaxes(title_text='jitter_max_frac')
fig.update_yaxes(title_text='metres', row=1, col=1)
fig.update_yaxes(title_text='metres', row=1, col=2)
fig.update_layout(
    title='Jitter sweep — co-location scenario (300 records, same shuffled tile)',
    template='plotly_white', height=420, showlegend=False,
)
fig.show()

**Figure 8d.** Line charts showing EDD (left) and MNND (right) as functions of `jitter_max_frac` (0.05–0.50), confirming that both displacement metrics scale approximately linearly with the jitter fraction parameter.

## Summary

| Metric | Reference point | Interpretation |
|--------|----------------|----------------|
| EDD | Shuffled tile centre → display pin | Scales linearly with J; bounded by J√2 |
| MNND (display) | Within co-located display set | Positional diversity; too small → fingerprinting risk |
| DBSCAN F1 | True clusters vs display clusters | Low F1 = PRP has destroyed spatial structure (privacy win) |

**Comparison to Lin (2023):** Lin's framework treats high F1 as good (utility preservation). In this library the PRP makes F1 low by design — the display tier should *not* reveal true cluster structure. EDD and MNND serve the same diagnostic role as in Lin, but are computed within the shuffled-tile space so they measure jitter alone.

**Tuning guidance:** `jitter_max_frac = 0.25` (J = 62.5 m on a 250 m tile) provides adequate positional diversity without pushing display pins past adjacent tile boundaries. Values above 1.0 (J ≥ bin size) may produce display artefacts where pins appear to belong to visually distant locations.

## References

- **Snow, J.** (1855). *On the Mode of Communication of Cholera* (2nd ed.). Churchill, London. — Source of the 1854 Soho cholera death and pump location dataset used throughout these notebooks.
- **Brody, H., Rip, M.R., Vinten-Johansen, P., Paneth, N., & Rachman, S.** (2000). Map-making and myth-making in Broad Street: the London cholera epidemic, 1854. *The Lancet, 356*(9223), 64–68.
- **Lin, Y.** (2023). Geo-indistinguishable masking: enhancing privacy protection in spatial point mapping. *Cartography and Geographic Information Science*. https://doi.org/10.1080/15230406.2023.2267967
- **Ester, M., Kriegel, H.-P., Sander, J., & Xu, X.** (1996). A density-based algorithm for discovering clusters in large spatial databases with noise. *Proceedings of KDD 1996*, 226–231. — DBSCAN algorithm used for cluster evaluation in NB08.

## Glossary

| Term | Definition |
|------|-----------|
| **Expected Displacement Distance (EDD)** | The mean haversine distance between each shuffled tile centre and its corresponding display pin; measures jitter spread in metres. |
| **Mean Nearest-Neighbour Distance (MNND)** | The average distance from each display pin to its closest neighbour; measures how spread out display pins are relative to tile centres. |
| **DBSCAN** | Density-Based Spatial Clustering of Applications with Noise; groups nearby points into clusters without requiring a pre-specified cluster count. |
| **Cluster fidelity** | The degree to which display-position clusters match the true geographic clusters; low fidelity (high privacy) means the PRP has destroyed cluster structure. |
| **Haversine distance** | The great-circle distance between two (lat, lon) points on a sphere; used here to measure displacement in metres. |
| **Geo-indistinguishable masking** | The privacy technique introduced by Lin (2023) that displaces spatial points to satisfy geo-indistinguishability; EDD and MNND are its primary evaluation metrics. |
| **Jitter quality** | How well the display jitter distributes pins within the expected radius around the tile centre; measured by EDD relative to the theoretical maximum `J√2`. |
| **Jitter sweep** | An experiment varying `jitter_max_frac` across a range to quantify how EDD and MNND scale with the jitter parameter. |
| **Catchment area** | The set of death locations assigned to a given pump by nearest-neighbour distance; used here to define natural geographic clusters in the cholera dataset. |